# Analyse du fonctionnement de pack()  
J'en arrive à la conclusion que pour recouvrir un frame contenant un ou plusieurs widgets, il faut forget() ces widgets. Dans le cas contraire, le frame contenant occupe sa place à parts égales avec le ou les frames qui prétendent le recouvrir. La solution consistant à forget() le frame contenant ne fonctionne pas, non plus, car les frames contenus n'ont plus de point d'attache et c'est le fond de la self qui apparaît.  
J'en déduis que pour une gestion fine des frames il faut avoir recours à grid ou place

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.geometry("1200x800")
root.title("Fenêtre test")


# --- Cadre 1 (au-dessus) ---
fr1_n1 = tk.Frame(root, bg="blue")
fr1_n1.pack(side=tk.TOP, fill="both", expand=True)

lb_fr1_n1 = tk.Label(fr1_n1, text="Label de fr1_n1", bg="darkorchid4")
lb_fr1_n1.pack(expand=True)

# --- Cadre 2 (en bas) ---
fr2_n1 = tk.Frame(root, height=100, bg="orange")
fr2_n1.pack(side=tk.BOTTOM, fill="x")
fr2_n1.pack_propagate(False)

fr1_n2 = tk.Frame(fr2_n1, height=20, bg="green")
fr1_n2.pack(side=tk.TOP, fill="x", padx=5, pady=5)

fr2_n2 = tk.Frame(fr2_n1, bg="red")
fr2_n2.pack(side=tk.BOTTOM, fill="both", expand=True)

# Empilement de Frames dans fr1_n1. Je les nomme cadre_x pour bien les suivre
cadre_1 = tk.Frame(fr1_n1, bg="AntiqueWhite2")
cadre_2 = tk.Frame(fr1_n1, bg="BlueViolet")
cadre_3 = tk.Frame(fr1_n1, bg="chartreuse3")
lb_fr1_n1.forget()
cadre_1.pack(side=tk.TOP, fill="both", expand=True)
# cadre_2.pack(side=tk.TOP, fill="both", expand=True)
# cadre_3.pack(side=tk.TOP, fill="both", expand=True)

root.mainloop()


# Analyse du fonctionnement de tk.grid  
Voir https://tkdocs.com/tutorial/grid.html 

In [19]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.geometry("1200x800")
root.title("Fenêtre test")

# --- Configuration de la Grille Racine (root) ---
# Ceci permet aux colonnes/lignes de root de se développer
root.grid_columnconfigure(0, weight=1) # La colonne 0 prend tout l'espace horizontal
root.grid_rowconfigure(0, weight=1)     # f1 prend de l'espace vertical
root.grid_rowconfigure(1, weight=1)     # f2 prend de l'espace vertical (partage équitable avec f1)

# --- Cadre 1 (au-dessus) ---
f1 = tk.Frame(root, bg="blue")
# 'sticky="nsew"' force f1 à remplir complètement sa cellule (ligne 0, colonne 0)
f1.grid(row=0, column=0, sticky="nsew") 

# Pour centrer le Label dans f1
f1.grid_columnconfigure(0, weight=1)
f1.grid_rowconfigure(0, weight=1)

# Label dans Cadre 1
lbl1_f1 = tk.Label(f1, text="Label de f1", bg="darkorchid4")
# Le label n'a pas besoin de sticky s'il est au centre
lbl1_f1.grid(row=0, column=0)


# --- Cadre 2 (en bas) ---
# Note : 'height=100' est maintenant moins pertinent car 'sticky' va le forcer à s'étirer.
f2 = tk.Frame(root, bg="orange")

# 'sticky="nsew"' force f2 à remplir complètement sa cellule (ligne 1, colonne 0)
f2.grid(row=1, column=0, sticky="nsew")

# --- Configuration de la Grille du Cadre f2 ---
# f2 contient f1_2 et f2_2. Nous devons lui dire comment ils doivent se développer :
f2.grid_columnconfigure(0, weight=1) # La seule colonne de f2 prend 100% de la largeur
f2.grid_rowconfigure(0, weight=1)     # f1_2 prend 50% de la hauteur
f2.grid_rowconfigure(1, weight=1)     # f2_2 prend 50% de la hauteur

# Cadre 1 dans Cadre_2 
f1_2 = tk.Frame(f2, bg="green")
# 'sticky="nsew"' force f1_2 à remplir complètement sa cellule dans f2
f1_2.grid(row=0, column=0, sticky="nsew",padx=10,pady=10) 

# Cadre 2 dans Cadre_2 
f2_2 = tk.Frame(f2, bg="red")
# 'sticky="nsew"' force f2_2 à remplir complètement sa cellule dans f2
f2_2.grid(row=1, column=0, sticky="nsew")
# Empilement de Frames dans fr1_n1. Je les nomme cadre_x pour bien les suivre
# cadre_1 = tk.Frame(f1, bg="AntiqueWhite2")
# cadre_2 = tk.Frame(f1, bg="BlueViolet")
# cadre_3 = tk.Frame(f1, bg="chartreuse3")

root.mainloop()

# Mise au point outil d'analyse des objets Tkinter

#### Le script

In [4]:
widget_names = {}

def set_widget_names(widget, name):
    widget_names[widget] = name
    return widget

def nom_widget(widget):
    """Retourne un nom sûr (set_widget_names() ou fallback), même widget détruit."""
    if widget in widget_names:
        return widget_names[widget]

    # widget détruit → on renvoie une string propre
    try:
        return widget.winfo_name()
    except:
        return "<destroyed>"

def safe_call(func, default=None):
    """Permet d'appeler winfo/pack/grid/place en sécurité."""
    try:
        return func()
    except:
        return default

def get_geometry_info(widget):
    """Retourne la géométrie (pack|grid|place) sans jamais planter."""
    
    info = safe_call(widget.pack_info)
    if info:
        return "pack", info

    info = safe_call(widget.grid_info)
    if info:
        return "grid", info

    info = safe_call(widget.place_info)
    if info:
        return "place", info

    return None, {}

def afficher_hierarchie(widget, indent=0):
    """Affiche la hiérarchie sans planter même si le widget est détruit."""

    prefix = "  " * indent

    nom = nom_widget(widget)
    classe = safe_call(widget.winfo_class, "<destroyed>")

    geom, params = get_geometry_info(widget)
    if geom:
        params_str = ", ".join(f"{k}={v}" for k, v in params.items())
        geom_str = f"{geom}({params_str})"
    else:
        geom_str = "no-layout"

    print(f"{prefix}- {nom} [{classe}] → {geom_str}")

    # Récupérer enfants en mode safe
    enfants = safe_call(widget.winfo_children, [])
    if not enfants:
        return

    for enfant in enfants:
        afficher_hierarchie(enfant, indent + 1)


#### L'objet à analyser

In [10]:
import yaml
import tkinter as tk
from tkinter import ttk
from pathlib import Path
from tkinter import TclError
from colorama import Fore, Style

ecran = None

class AppUi(tk.Tk):
    nb_instances = 0

    def __init__(self):
        # *******************
        AppUi.nb_instances += 1
        if AppUi.nb_instances > 1:
            print(
                Fore.RED + f"Nb instances AppUi = {AppUi.nb_instances}" + Style.RESET_ALL)
        # *******************
        super().__init__()

        self.geometry("1200x800")
        self.title("Ecran de mon application")

        # Instanciation du manager de widgets (self.wtm_xxx).
        # self.wtm = afficher_hierarchie(self)

        # Création de la racine du menu principal
        self.menubar = tk.Menu(self)
        self.configure(menu=self.menubar)
        set_widget_names(self.menubar, "menubar")
        # breakpoint()

        # Création et paramétrage d'un objet ttk.Style
        style = ttk.Style()
        style.configure('Blue.TFrame',background='blue')
        style.configure('Red.TFrame',background='red')  
              
        # Création d'un Frame central pour accueillir les pages
        self.fr_centre = ttk.Frame(self,style='Blue.TFrame')
        self.fr_centre.grid(column=0,row=0,sticky="nsew")
        # Caractéristique de la grille (0,0)
        self.columnconfigure(0, weight=1)
        self.rowconfigure(0, weight=1)
        # Pour permettre l'analyse ultérieure du widget
        set_widget_names(self.fr_centre, "fr_centre")

        # Création du frame fr_statut
        self.fr_statut = ttk.Frame(self,height=100,style='Red.TFrame')
        self.fr_statut.grid(column=0,row=1,sticky="ew")
        # Caractéristique de la grille (1,0)
        self.rowconfigure(1, weight=0)
        # Pour permettre l'analyse ultérieure du widget
        set_widget_names(self.fr_statut, "fr_centre")
    

        


ecran = AppUi()
ecran.mainloop()


#### L'analyse

In [27]:
print(type(root))
# afficher_hierarchie(root)
root.mainloop()

<class 'tkinter.Tk'>
